# Where does EEG + text sentiment fail? — diagnostics

Four stages follow the information from the brain to the prediction (see `reports/diagnostics_plan.md`):

1. **Signal** — is reliable, sentiment-related information in word-level EEG? (variance components, cross-reader reliability, probes with positive controls: word length, frequency, surprisal, reading time, word class)
2. **Representation** — which sentence-level representation keeps it? (word EEG, NeuroLM, handcrafted)
3. **Fusion** — does the trained LoRA model read its EEG tokens? (attention, attribution, counterfactual EEG swaps, probes of EEG-slot states)
4. **Errors** — where does the text model fail, and does EEG help there? Can EEG predict text-model errors?

Stages 1, 2, 4 run on CPU; training and stage 3 need a GPU. Everything is saved to Drive.

## 1. Repository and dependencies

In [ ]:
import os
REPO_URL = 'https://github.com/MohammadJRanjbar/zuco-multimodal-sentiment.git'
BRANCH = 'feature/neurolm-eeg-probe'
REPO_DIR = '/content/zuco-multimodal-sentiment'
if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin && git -C $REPO_DIR checkout $BRANCH && git -C $REPO_DIR -c user.name=colab -c user.email=colab@users.noreply.github.com pull --rebase origin $BRANCH
%cd $REPO_DIR
!pip install -q -r requirements-diagnostics.txt
!git log --oneline -1

## 2. Drive and paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
THESIS_ROOT = '/content/drive/MyDrive/Thesis'
MAT_DIR = f'{THESIS_ROOT}/Data/zuco_og_raw'
LABELS_CSV = f'{THESIS_ROOT}/Data/zuco_sentiment_labels_task1_fixed.csv'
ARTIFACTS = f'{THESIS_ROOT}/CachedArtifacts/zuco_multimodal_sentiment'
WORD_EEG_DIR = f'{ARTIFACTS}/word_eeg/TRT'
HANDCRAFTED_DIR = f'{ARTIFACTS}/eeg_features/classical_v2_normalized_line_length'
NEUROLM_CACHE_DIR = f'{ARTIFACTS}/neurolm_embeddings'
LORA_RESULTS = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment/eeg_text_lora'
LORA_RUN_TAG = 'v1_word_aligned'
DIAG = f'{THESIS_ROOT}/Results/zuco_multimodal_sentiment/diagnostics/v1'
os.environ['HF_HOME'] = f'{ARTIFACTS}/hf_cache'
os.makedirs(DIAG, exist_ok=True)
DEVICE = 'cuda' if os.system('nvidia-smi > /dev/null 2>&1') == 0 else 'cpu'
EXTRA = ''
if os.path.exists(NEUROLM_CACHE_DIR):
    EXTRA += f' --neurolm-cache-dir "{NEUROLM_CACHE_DIR}"'
if os.path.exists(HANDCRAFTED_DIR):
    EXTRA += f' --handcrafted-dir "{HANDCRAFTED_DIR}"'
print('device:', DEVICE, '| stage-2 extras:', EXTRA or 'none')

## 3. Tests

In [ ]:
!python -m pytest -q tests/test_diagnostics.py tests/test_fusion.py tests/test_word_eeg.py

## 4. Word-level EEG (once)

If you extracted it before reading times were added, set `REEXTRACT = True` once.

In [ ]:
REEXTRACT = False
LOCAL_MAT = '/content/zuco_og_raw'
!mkdir -p $LOCAL_MAT && rsync -a --include='results*_SR.mat' --exclude='*' "$MAT_DIR/" "$LOCAL_MAT/"
flag = '--overwrite' if REEXTRACT else ''
import glob, h5py
for path in sorted(glob.glob(f'{LOCAL_MAT if "LOCAL_MAT" in dir() else MAT_DIR}/results*_SR.mat')):
    print(os.path.basename(path), 'HDF5 (v7.3)' if h5py.is_hdf5(path) else 'MATLAB v5', f'{os.path.getsize(path) / 1e9:.2f} GB')
!python scripts/extract_word_eeg.py --mat-dir "$LOCAL_MAT" --labels-csv "$LABELS_CSV" --out-dir "$WORD_EEG_DIR" $flag

## 5. Stages 1–2: what word-level EEG encodes, and which representation keeps it

In [ ]:
!python scripts/analyze_eeg_signal.py --word-eeg-dir "$WORD_EEG_DIR" --labels-csv "$LABELS_CSV" \
    --out-dir "$DIAG/stage1" --device $DEVICE --n-perm 50 --n-perm-single 20 $EXTRA

## 6. Train the EEG + text LoRA model with saved weights (GPU)

Trains fold 1 for all four arms (`FOLDS = 1`); set `FOLDS = 5` later and finished folds are reused. `--retrain-missing-weights` retrains only folds trained before weights were saved. Stage 3 needs `text_eeg` fold 0; the stage-4 comparisons need all arms.

In [ ]:
FOLDS = 1  # start with one fold; raise to 5 later (finished folds are reused, not retrained)
!python scripts/run_eeg_text_lora.py --word-eeg-dir "$WORD_EEG_DIR" --results-dir "$LORA_RESULTS" \
    --run-tag $LORA_RUN_TAG --retrain-missing-weights --folds $FOLDS


## 7. Stage 3: inside the trained model (GPU)

In [ ]:
!python scripts/analyze_fusion_model.py --word-eeg-dir "$WORD_EEG_DIR" --lora-run-dir "$LORA_RESULTS/$LORA_RUN_TAG" \
    --word-targets "$DIAG/stage1/word_targets.csv" --out-dir "$DIAG/stage3" --arm text_eeg --fold 0 --max-trials 400

## 8. Stage 4: where the text model fails

In [ ]:
!python scripts/analyze_errors.py --lora-run-dir "$LORA_RESULTS/$LORA_RUN_TAG" --labels-csv "$LABELS_CSV" \
    --word-eeg-dir "$WORD_EEG_DIR" --out-dir "$DIAG/stage4"

## 9. Report

In [ ]:
!python scripts/build_diagnostics_report.py --signal-dir "$DIAG/stage1" --fusion-dir "$DIAG/stage3" \
    --errors-dir "$DIAG/stage4" --lora-report "$LORA_RESULTS/$LORA_RUN_TAG/eeg_text_lora_results.md" \
    --report reports/eeg_text_diagnostics.md
!cp reports/eeg_text_diagnostics.md "$DIAG/"
from IPython.display import Image, Markdown, display
display(Markdown(open('reports/eeg_text_diagnostics.md').read()))
for name in ['stage1/plots/reliability_all_readers.png', 'stage1/plots/correlation_surprisal.png',
             'stage1/plots/correlation_valence.png', 'stage3/attention_by_layer.png']:
    path = f'{DIAG}/{name}'
    if os.path.exists(path):
        display(Image(path))

## Save results to GitHub

Small files (reports, summaries, predictions, plots) are committed to `saved_results/` on your fork; trained weights and caches stay on Drive and are listed with sizes and SHA-256 in `LARGE_FILES.md`.

**One-time setup:** on GitHub create a fine-grained token (Settings → Developer settings → Personal access tokens) with *Contents: Read and write* for `MohammadJRanjbar/zuco-multimodal-sentiment`; in Colab open the key icon (Secrets), add it as `GITHUB_TOKEN`, and allow this notebook to access it.

In [ ]:
from google.colab import userdata
os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
GIT_NAME, GIT_EMAIL = 'mohammadranjbar', 'mjrkalhar@gmail.com'
!python scripts/save_results_to_github.py --name diagnostics_v1 --source "$DIAG" "$LORA_RESULTS/$LORA_RUN_TAG" --push \
    --author-name "$GIT_NAME" --author-email "$GIT_EMAIL"
